In [ ]:
import pandas as pd

# === Step 1: Load all files ===
patients = pd.read_csv("patients.csv")
encounters = pd.read_csv("encounters.csv")
observations = pd.read_csv("observations.csv")
conditions = pd.read_csv("conditions.csv")  # You forgot this in your code

# === Step 2: Prepare patient demographics ===
patients['age'] = pd.to_datetime("2025-01-01") - pd.to_datetime(patients["BIRTHDATE"])
patients['age'] = patients['age'].dt.days // 365
patients_base = patients[["Id", "age", "GENDER"]].rename(columns={"Id": "PATIENT"})

# === Step 3: Aggregate all conditions per patient ===
cond_summary = conditions.groupby("PATIENT")["DESCRIPTION"]\
    .apply(lambda x: ", ".join(sorted(set(x)))).reset_index()

# === Step 4: Filter only vital signs, then pivot latest values ===
observations["DATE"] = pd.to_datetime(observations["DATE"])
vitals_only = observations[observations["TYPE"] == "vital-signs"]

obs_latest = (
    vitals_only.sort_values("DATE")
    .groupby(["PATIENT", "DESCRIPTION"])
    .tail(1)
    .pivot(index="PATIENT", columns="DESCRIPTION", values="VALUE")
    .reset_index()
)

# === Step 5: Get latest encounter ID and class per patient ===
latest_encounters = (
    encounters.sort_values("START")
    .groupby("PATIENT")
    .tail(1)[["PATIENT", "Id", "ENCOUNTERCLASS"]]
    .rename(columns={"Id": "ENCOUNTER"})
)

# === Step 6: Merge all components into final dataset ===
df = patients_base\
    .merge(cond_summary, on="PATIENT", how="left")\
    .merge(obs_latest, on="PATIENT", how="left")\
    .merge(latest_encounters, on="PATIENT", how="left")

# === Step 7: Save ===
df.to_csv("final_patient_dataset_with_vital_signs_only.csv", index=False)
print("Saved as final_patient_dataset_with_vital_signs_only.csv")

Saved as final_patient_dataset_with_vital_signs_only.csv
